# Faz 0.2 (v3) — Memory-Efficient Sampling + HF Upload

**v3'te düzeltilen sorunlar:**
- `import json` ve diğer importlar EN BAŞTA → NameError fix
- HF upload artık 3 ayrı config olarak (`interactions`, `items`, `cold_start`) → schema cast hatası fix
- **Resumable**: Drive'da parquet'lar varsa sampling atlanır, doğrudan upload'a geçilir
- Eski bozuk HF repo otomatik silinir

**Beklenen peak RAM:** ~4-6 GB (Colab free 12 GB'da rahat)

**Eğer sampling zaten yapıldıysa (parquet'lar Drive'da):** Section 3 otomatik atlar, ~2 dakikada bitirir.

## 1. Tüm Importlar (NameError'lara karşı sigorta)

In [ ]:
# Bütün importları başta yapıyoruz ki herhangi bir hücrede NameError olmasın
import os
import sys
import json
import gc
import time
import random

import numpy as np
import pandas as pd

In [ ]:
# Paketler (datasets versiyonu sabitleniyor — yenisi schema-cast'ı sıkı tutuyor)
!pip install -q "datasets<3.0" pandas pyarrow huggingface_hub
print("✓ Paketler hazır. Eğer ilk seferse: Runtime → Restart session → bu hücreden devam.")

In [ ]:
# Restart sonrası tekrar import (Colab restart'ta sys.modules sıfırlanır)
import os, sys, json, gc, time, random
import numpy as np
import pandas as pd

from datasets import Dataset, load_dataset
from huggingface_hub import HfApi, create_repo, login, whoami

print("✓ Tüm modüller import edildi")

## 2. Konfigürasyon

In [ ]:
CONFIG = {
    'date_start': '2018-01-01',
    'date_end':   '2023-09-12',
    'k_core': 5,
    'cold_start_n_users': 5000,
    'cold_start_min_reviews': 1,
    'cold_start_max_reviews': 2,
    'hf_dataset_name': 'amazon-musical-instruments-2018-2023-5core',
    'hf_private': False,
    'seed': 42,
    # True yaparsan parquet cache olsa bile sampling baştan yapılır
    'force_rebuild': False,
}

np.random.seed(CONFIG['seed'])
random.seed(CONFIG['seed'])
print('CONFIG:', json.dumps(CONFIG, indent=2))

## 3. Drive Mount + Cache Kontrolü

Eğer `interactions.parquet`, `cold_start_interactions.parquet`, `items.parquet` Drive'da varsa sampling adımları **atlanır**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/data-mining-hw'
SAMPLED_DIR = f'{PROJECT_DIR}/data/sampled'
os.makedirs(SAMPLED_DIR, exist_ok=True)
print(f'✓ Drive hazır: {SAMPLED_DIR}')

In [ ]:
required_files = ['interactions.parquet', 'cold_start_interactions.parquet', 'items.parquet']
existing = set(os.listdir(SAMPLED_DIR))
all_present = all(f in existing for f in required_files)

USE_CACHE = all_present and not CONFIG['force_rebuild']

if all_present:
    print('✓ Cache bulundu:')
    for fn in required_files:
        size_mb = os.path.getsize(f'{SAMPLED_DIR}/{fn}') / 1e6
        print(f'  {fn}: {size_mb:.2f} MB')
    if CONFIG['force_rebuild']:
        print('\n⚠ force_rebuild=True → sampling baştan yapılacak')
    else:
        print('\n→ Sampling atlanacak, doğrudan upload\'a geçilecek')
else:
    missing = [f for f in required_files if f not in existing]
    print(f'✗ Cache eksik: {missing}')
    print('→ Sampling adımları çalışacak')

## 4. HuggingFace Login (Write yetkili token gerekli)

In [ ]:
token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception as e:
    print(f'Secret okunamadı: {e}')

if not token:
    raise RuntimeError('HF_TOKEN gerekli. Sol panel → 🔑 → HF_TOKEN ekle (Write yetkili, Notebook access açık)')

login(token=token)
HF_USERNAME = whoami()['name']
REPO_ID = f"{HF_USERNAME}/{CONFIG['hf_dataset_name']}"
print(f'✓ HF login: {HF_USERNAME}')
print(f'✓ Hedef repo: {REPO_ID}')

## 5. Sampling (sadece cache yoksa çalışır)

`USE_CACHE=True` ise bu blok hiçbir şey yapmaz, doğrudan section 6'ya geç.

In [ ]:
if USE_CACHE:
    print('→ Cache mevcut, sampling atlanıyor')
else:
    print('Sampling başlıyor (~10-15 dk)')
    print('Reviews yükleniyor (cache\'den okur)...')
    t0 = time.time()
    ds_reviews = load_dataset(
        'McAuley-Lab/Amazon-Reviews-2023',
        'raw_review_Musical_Instruments',
        split='full',
        trust_remote_code=True
    )
    print(f'✓ Reviews: {len(ds_reviews):,} satır ({time.time()-t0:.1f}s)')

    t0 = time.time()
    print('Metadata yükleniyor...')
    ds_meta = load_dataset(
        'McAuley-Lab/Amazon-Reviews-2023',
        'raw_meta_Musical_Instruments',
        split='full',
        trust_remote_code=True
    )
    print(f'✓ Metadata: {len(ds_meta):,} satır ({time.time()-t0:.1f}s)')

In [ ]:
if not USE_CACHE:
    # Time filter
    start_ts = int(pd.Timestamp(CONFIG['date_start']).timestamp() * 1000)
    end_ts   = int(pd.Timestamp(CONFIG['date_end']).timestamp() * 1000)
    print(f'Time filter: {CONFIG["date_start"]} → {CONFIG["date_end"]}')

    t0 = time.time()
    ds_time = ds_reviews.filter(
        lambda batch: [(ts >= start_ts) and (ts <= end_ts) for ts in batch['timestamp']],
        batched=True, batch_size=50000,
    )
    print(f'✓ {len(ds_reviews):,} → {len(ds_time):,} ({time.time()-t0:.1f}s)')

    # Hafif pandas (sadece ID/timestamp/rating)
    print('Hafif pandas oluşturuluyor...')
    t0 = time.time()
    ds_light = ds_time.select_columns([
        'user_id', 'parent_asin', 'rating',
        'timestamp', 'helpful_vote', 'verified_purchase'
    ])
    df_light = ds_light.to_pandas()
    print(f'✓ Shape: {df_light.shape}, bellek: {df_light.memory_usage(deep=True).sum()/1e6:.1f} MB ({time.time()-t0:.1f}s)')

    del ds_light
    gc.collect()

In [ ]:
if not USE_CACHE:
    # Cold-start ayır
    user_counts = df_light['user_id'].value_counts()
    cold_candidates = user_counts[
        (user_counts >= CONFIG['cold_start_min_reviews']) &
        (user_counts <= CONFIG['cold_start_max_reviews'])
    ].index.tolist()
    print(f'Cold-start aday havuzu: {len(cold_candidates):,} kullanıcı')

    np.random.seed(CONFIG['seed'])
    cold_users = set(np.random.choice(
        cold_candidates,
        size=min(CONFIG['cold_start_n_users'], len(cold_candidates)),
        replace=False
    ).tolist())
    print(f'Seçilen cold-start: {len(cold_users):,}')

    df_light = df_light[~df_light['user_id'].isin(cold_users)].copy()
    print(f'5-core için kalan: {len(df_light):,}')
    gc.collect()

In [ ]:
if not USE_CACHE:
    # İteratif 5-core
    def iterative_k_core(df, k=5, user_col='user_id', item_col='parent_asin', max_iter=20):
        print(f'İteratif {k}-core başlıyor (başlangıç: {len(df):,})')
        for i in range(max_iter):
            n_before = len(df)
            uc = df[user_col].value_counts()
            df = df[df[user_col].isin(uc[uc >= k].index)]
            ic = df[item_col].value_counts()
            df = df[df[item_col].isin(ic[ic >= k].index)]
            n_users = df[user_col].nunique()
            n_items = df[item_col].nunique()
            print(f'  Iter {i+1}: {n_before:,} → {len(df):,} ({n_users:,} U / {n_items:,} I)')
            if len(df) == n_before:
                print(f'  ✓ Stabil (iter {i+1})')
                break
        return df

    df_light = iterative_k_core(df_light, k=CONFIG['k_core'])
    final_users = set(df_light['user_id'].unique())
    final_items = set(df_light['parent_asin'].unique())
    print(f'\n✓ Main set: {len(df_light):,} interaction, {len(final_users):,} U, {len(final_items):,} I')

In [ ]:
if not USE_CACHE:
    # Şimdi text dahil filtrele
    all_users = final_users | cold_users
    print('HF dataset final ID\'lere göre filtreleniyor...')
    t0 = time.time()
    ds_final = ds_time.filter(
        lambda batch: [
            (uid in all_users) and (iid in final_items if uid in final_users else True)
            for uid, iid in zip(batch['user_id'], batch['parent_asin'])
        ],
        batched=True, batch_size=50000,
    )
    print(f'✓ {len(ds_time):,} → {len(ds_final):,} ({time.time()-t0:.1f}s)')

    del ds_reviews, ds_time, df_light
    gc.collect()

    # Text dahil pandas'a çevir
    print('Final dataset pandas\'a çevriliyor (text dahil)...')
    t0 = time.time()
    df_full = ds_final.to_pandas()
    print(f'✓ Shape: {df_full.shape}, bellek: {df_full.memory_usage(deep=True).sum()/1e9:.2f} GB ({time.time()-t0:.1f}s)')
    del ds_final
    gc.collect()

In [ ]:
if not USE_CACHE:
    # Main vs cold-start ayır
    mask_main = df_full['user_id'].isin(final_users)
    mask_cold = df_full['user_id'].isin(cold_users)
    df_main = df_full[mask_main].copy()
    df_cold = df_full[mask_cold].copy()

    before_cold = len(df_cold)
    df_cold = df_cold[df_cold['parent_asin'].isin(final_items)].copy()
    print(f'Cold-start filtre: {before_cold:,} → {len(df_cold):,} (sadece main\'de olan ürünler)')

    del df_full
    gc.collect()

    def prep(df):
        df = df[['user_id', 'parent_asin', 'rating', 'timestamp',
                 'helpful_vote', 'verified_purchase', 'title', 'text']].copy()
        df = df.rename(columns={'title': 'review_title', 'text': 'review_text'})
        df['date'] = pd.to_datetime(df['timestamp'], unit='ms')
        return df

    df_main = prep(df_main)
    df_cold = prep(df_cold)
    print(f'\n✓ Main: {df_main.shape}')
    print(f'✓ Cold-start: {df_cold.shape}')

In [ ]:
if not USE_CACHE:
    # Metadata işle
    print('Metadata filtreleniyor...')
    t0 = time.time()
    ds_meta_filtered = ds_meta.filter(
        lambda batch: [pa in final_items for pa in batch['parent_asin']],
        batched=True, batch_size=50000,
    )
    print(f'✓ {len(ds_meta):,} → {len(ds_meta_filtered):,} ({time.time()-t0:.1f}s)')
    del ds_meta
    gc.collect()

    df_meta_raw = ds_meta_filtered.to_pandas()
    print(f'Meta pandas: {df_meta_raw.shape}, {df_meta_raw.memory_usage(deep=True).sum()/1e6:.1f} MB')
    del ds_meta_filtered
    gc.collect()

    # Düzleştir
    def extract_first_large_image(img):
        try:
            if isinstance(img, dict):
                large = img.get('large', [])
                if isinstance(large, np.ndarray): large = large.tolist()
                if large and len(large) > 0:
                    return large[0]
        except: pass
        return None

    def join_list_field(x, sep=' '):
        if isinstance(x, (list, np.ndarray)):
            return sep.join([str(s) for s in x if s])
        return str(x) if x else ''

    def parse_price(p):
        if p in [None, 'None', '', 'nan']: return None
        try:
            return float(str(p).replace('$', '').replace(',', '').strip())
        except: return None

    df_items = pd.DataFrame({
        'parent_asin':    df_meta_raw['parent_asin'].values,
        'title':          df_meta_raw['title'].values,
        'description':    [join_list_field(d, ' ') for d in df_meta_raw['description']],
        'features':       [join_list_field(f, ' | ') for f in df_meta_raw['features']],
        'main_category':  df_meta_raw['main_category'].values,
        'categories':     [join_list_field(c, ' > ') for c in df_meta_raw['categories']],
        'store':          df_meta_raw['store'].values,
        'price':          [parse_price(p) for p in df_meta_raw['price']],
        'average_rating': df_meta_raw['average_rating'].values,
        'rating_number':  df_meta_raw['rating_number'].values,
        'image_url':      [extract_first_large_image(img) for img in df_meta_raw['images']],
    })
    df_items['has_image']       = df_items['image_url'].notna()
    df_items['has_description'] = df_items['description'].str.len() > 10
    df_items['has_price']       = df_items['price'].notna()

    del df_meta_raw
    gc.collect()
    print(f'\n✓ Items: {df_items.shape}')
    print(f'  Görseli olan:    {df_items["has_image"].sum():,} ({df_items["has_image"].mean()*100:.1f}%)')
    print(f'  Açıklaması olan: {df_items["has_description"].sum():,} ({df_items["has_description"].mean()*100:.1f}%)')
    print(f'  Fiyatı olan:     {df_items["has_price"].sum():,} ({df_items["has_price"].mean()*100:.1f}%)')

In [ ]:
if not USE_CACHE:
    # Drive'a kaydet
    df_main.to_parquet(f'{SAMPLED_DIR}/interactions.parquet', index=False)
    df_cold.to_parquet(f'{SAMPLED_DIR}/cold_start_interactions.parquet', index=False)
    df_items.to_parquet(f'{SAMPLED_DIR}/items.parquet', index=False)
    print('✓ Parquet\'lar kaydedildi:')
    for fn in ['interactions.parquet', 'cold_start_interactions.parquet', 'items.parquet']:
        size_mb = os.path.getsize(f'{SAMPLED_DIR}/{fn}') / 1e6
        print(f'  {fn}: {size_mb:.2f} MB')

## 6. Parquet'lardan Tek Tip Yükleme

Sampling yapıldı veya cache mevcut — fark etmez, buradan itibaren her şey aynı.

In [ ]:
df_main  = pd.read_parquet(f'{SAMPLED_DIR}/interactions.parquet')
df_cold  = pd.read_parquet(f'{SAMPLED_DIR}/cold_start_interactions.parquet')
df_items = pd.read_parquet(f'{SAMPLED_DIR}/items.parquet')

print(f'✓ df_main:  {df_main.shape}')
print(f'✓ df_cold:  {df_cold.shape}')
print(f'✓ df_items: {df_items.shape}')

print(f'\nMain kolonları:  {list(df_main.columns)}')
print(f'Items kolonları: {list(df_items.columns)}')

## 7. İstatistikler + README

In [ ]:
# json BURADA da yeniden import (güvenlik için, restart ihtimaline karşı)
import json

stats = {
    'category': 'Musical_Instruments',
    'sampling': {k: v for k, v in CONFIG.items() if k != 'force_rebuild'},
    'main_dataset': {
        'n_interactions': int(len(df_main)),
        'n_users': int(df_main['user_id'].nunique()),
        'n_items': int(df_main['parent_asin'].nunique()),
        'date_min': str(pd.to_datetime(df_main['timestamp'], unit='ms').min().date()),
        'date_max': str(pd.to_datetime(df_main['timestamp'], unit='ms').max().date()),
        'mean_rating': float(df_main['rating'].mean()),
        'sparsity': float(1 - len(df_main) / (df_main['user_id'].nunique() * df_main['parent_asin'].nunique())),
    },
    'cold_start': {
        'n_interactions': int(len(df_cold)),
        'n_users': int(df_cold['user_id'].nunique()),
        'n_items': int(df_cold['parent_asin'].nunique()),
    },
    'items': {
        'total': int(len(df_items)),
        'with_image_pct': float(df_items['has_image'].mean()),
        'with_description_pct': float(df_items['has_description'].mean()),
        'with_price_pct': float(df_items['has_price'].mean()),
    },
}

with open(f'{SAMPLED_DIR}/stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print('✓ stats.json kaydedildi')
print('\n=== STATS ===')
print(json.dumps(stats, indent=2))

In [ ]:
readme = f'''---
license: mit
task_categories:
  - text-classification
  - text-retrieval
language:
  - en
tags:
  - recommendation-system
  - collaborative-filtering
  - amazon-reviews
  - musical-instruments
size_categories:
  - 100K<n<1M
configs:
  - config_name: interactions
    data_files:
      - split: train
        path: interactions/train-*
  - config_name: items
    data_files:
      - split: train
        path: items/train-*
  - config_name: cold_start
    data_files:
      - split: train
        path: cold_start/train-*
---

# Amazon Musical Instruments — 5-Core Sampled Dataset (2018-2023)

Sampled subset of [Amazon Reviews 2023](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023) Musical Instruments category, created for a Data Mining course recommendation system project.

## Sampling Strategy

1. **Time filter:** {CONFIG['date_start']} to {CONFIG['date_end']}
2. **Iterative {CONFIG['k_core']}-core filtering:** Both users and items have at least {CONFIG['k_core']} interactions (iteratively until stable)
3. **Cold-start test set:** {CONFIG['cold_start_n_users']} users with 1-2 reviews held out for cold-start evaluation
4. **Random seed:** {CONFIG['seed']}

## Configs

This dataset has **three configs**. Load each one separately:

```python
from datasets import load_dataset

interactions = load_dataset("{REPO_ID}", "interactions", split="train")
items        = load_dataset("{REPO_ID}", "items",        split="train")
cold_start   = load_dataset("{REPO_ID}", "cold_start",   split="train")
```

## Statistics

| Field | Value |
|---|---|
| Users (main)        | {stats["main_dataset"]["n_users"]:,} |
| Items (main)        | {stats["main_dataset"]["n_items"]:,} |
| Interactions (main) | {stats["main_dataset"]["n_interactions"]:,} |
| Sparsity            | {stats["main_dataset"]["sparsity"]:.6f} |
| Mean rating         | {stats["main_dataset"]["mean_rating"]:.3f} |
| Items w/ image      | {stats["items"]["with_image_pct"]*100:.1f}% |
| Items w/ description| {stats["items"]["with_description_pct"]*100:.1f}% |
| Items w/ price      | {stats["items"]["with_price_pct"]*100:.1f}% |
| Cold-start users    | {stats["cold_start"]["n_users"]:,} |

## Schema

### interactions / cold_start
- `user_id`, `parent_asin`, `rating`, `timestamp` (ms), `date`
- `helpful_vote`, `verified_purchase`
- `review_title`, `review_text`

### items
- `parent_asin`, `title`, `description`, `features`
- `main_category`, `categories`, `store`
- `price` (nullable), `average_rating`, `rating_number`
- `image_url` (nullable), `has_image`, `has_description`, `has_price`
'''

with open(f'{SAMPLED_DIR}/README.md', 'w') as f:
    f.write(readme)
print('✓ README.md hazır')

## 8. HF Upload — Bu kez doğru: 3 ayrı config olarak

**v2'nin hatası:** `push_to_hub` üç pandas'ı aynı config'e üç split olarak yüklemişti → HF bunları aynı şemada bekledi → CastError.

**v3'ün çözümü:** Her tablo kendi `config_name`'ine sahip → şemalar bağımsız.

In [ ]:
# Eski bozuk repo varsa sil (temiz başlangıç için)
api = HfApi()
try:
    api.delete_repo(REPO_ID, repo_type='dataset')
    print(f'✓ Eski bozuk repo silindi: {REPO_ID}')
except Exception as e:
    print(f'(Silme atlandı — repo zaten yok olabilir: {type(e).__name__})')

# Yeni repo
create_repo(repo_id=REPO_ID, repo_type='dataset', private=CONFIG['hf_private'], exist_ok=True)
print(f'✓ Yeni repo hazır: https://huggingface.co/datasets/{REPO_ID}')

In [ ]:
# Tablo 1: interactions
print('→ interactions yükleniyor...')
t0 = time.time()
ds_interactions = Dataset.from_pandas(df_main.reset_index(drop=True), preserve_index=False)
ds_interactions.push_to_hub(REPO_ID, config_name='interactions', private=CONFIG['hf_private'])
print(f'✓ interactions: {len(ds_interactions):,} satır ({time.time()-t0:.1f}s)')

In [ ]:
# Tablo 2: items
print('→ items yükleniyor...')
t0 = time.time()
ds_items_hf = Dataset.from_pandas(df_items.reset_index(drop=True), preserve_index=False)
ds_items_hf.push_to_hub(REPO_ID, config_name='items', private=CONFIG['hf_private'])
print(f'✓ items: {len(ds_items_hf):,} satır ({time.time()-t0:.1f}s)')

In [ ]:
# Tablo 3: cold_start
print('→ cold_start yükleniyor...')
t0 = time.time()
ds_cold = Dataset.from_pandas(df_cold.reset_index(drop=True), preserve_index=False)
ds_cold.push_to_hub(REPO_ID, config_name='cold_start', private=CONFIG['hf_private'])
print(f'✓ cold_start: {len(ds_cold):,} satır ({time.time()-t0:.1f}s)')

In [ ]:
# README'yi de upload et
api.upload_file(
    path_or_fileobj=f'{SAMPLED_DIR}/README.md',
    path_in_repo='README.md',
    repo_id=REPO_ID,
    repo_type='dataset',
)
print(f'✓ README yüklendi')
print(f'\n🎉 Dataset hazır: https://huggingface.co/datasets/{REPO_ID}')

## 9. Readback Test — Upload doğru çalıştı mı?

Üç config'i de HF'ten geri okuyup boyutlar eşleşiyor mu kontrol edelim.

In [ ]:
# Cache'i temizle ki gerçekten HF'ten okusun
import shutil
hf_cache = os.path.expanduser('~/.cache/huggingface/datasets')
if os.path.exists(hf_cache):
    for d in os.listdir(hf_cache):
        if CONFIG['hf_dataset_name'].replace('-', '_') in d.lower():
            shutil.rmtree(f'{hf_cache}/{d}', ignore_errors=True)
            print(f'  Cache silindi: {d}')

print('→ HF\'ten geri okunuyor...')
ds_check_int   = load_dataset(REPO_ID, 'interactions', split='train')
ds_check_items = load_dataset(REPO_ID, 'items',        split='train')
ds_check_cold  = load_dataset(REPO_ID, 'cold_start',   split='train')

print('\n=== READBACK ===')
ok1 = len(ds_check_int)   == len(df_main)
ok2 = len(ds_check_items) == len(df_items)
ok3 = len(ds_check_cold)  == len(df_cold)
print(f'interactions: {len(ds_check_int):,} (beklenen: {len(df_main):,}) {"✓" if ok1 else "✗"}')
print(f'items:        {len(ds_check_items):,} (beklenen: {len(df_items):,}) {"✓" if ok2 else "✗"}')
print(f'cold_start:   {len(ds_check_cold):,} (beklenen: {len(df_cold):,}) {"✓" if ok3 else "✗"}')

print(f'\ninteractions kolonları: {ds_check_int.column_names}')
print(f'items kolonları:        {ds_check_items.column_names}')

if ok1 and ok2 and ok3:
    print('\n🎉 HEPSİ DOĞRU. Kendi ortamımıza geçebiliriz.')
else:
    print('\n⚠ Uyuşmazlık var, yukarıdaki çıktıyı paylaş.')

In [ ]:
# Tek bir örneği yazdır - şemanın sağlamlığını teyit
print('=== ÖRNEK interaction ===')
print(ds_check_int[0])
print('\n=== ÖRNEK item (kısa alanlar) ===')
print({k: v for k, v in ds_check_items[0].items() if not isinstance(v, str) or len(v) < 100})

## 10. Özet

Eğer readback test'te üç ✓ gördüysen:

1. ✓ Sampling tamamlandı (parquet'lar Drive'da, HF'te)
2. ✓ HF dataset 3 config'le yapılandırıldı
3. ✓ Kendi ortamımıza geçip `src/data_utils.py`'ı güncelleyebiliriz

**Sıradaki adım:** Lokal ortama geç, `src/data_utils.py`'ı 3 config'i okuyacak şekilde güncelle, sonra `03_full_eda.ipynb`'e başla.